# AI-generated text detection

In this notebook, we train a language model to find out whether a given English answer was written by a human or by ChatGPT.

We use the HC3 dataset and compare three models: 

- A majority baseline
- TF-IDF with logistic regression
- Fine-tuned DistilBERT

Our main metric is macro-average F1.

**Research question:** We try to address this research question:

> Does fine-tuned DistilBERT get a higher macro-average F1 than TF-IDF logistic regression?

## 1. Imports and settings

In [ ]:
# Install pandas if it is missing, because the NLP VM does not include it by default.
try:
    import pandas as pd
except ImportError:
    %pip install -q pandas
    import pandas as pd

In [ ]:
# Import necessary libraries
import time

import copy
import json
import os
import random
import unicodedata
from datetime import datetime
from pathlib import Path

# We turn off tokenizer parallelism to avoid warnings in the notebook.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download, snapshot_download
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
# Define the class names
class_names = ["human-written", "AI-generated"]

# We use one fixed color for each class and each model in the plots.
class_colors = ["tab:blue", "tab:orange"]
model_colors = {
    "Majority baseline": "gray",
    "TF-IDF logistic regression": "tab:blue",
    "DistilBERT": "tab:orange",
}
training_color = "tab:blue"
validation_color = "tab:orange"

In [ ]:
# Experiment settings
SEED = 42
number_of_questions = 2500

# Total number of unseen questions to be added tothe test set only
extra_test_questions = 750

# Set the number of input to the model. 
# Originally, the number was 512, but we reduce it to 256 to save memory and speed up training
maximum_input_tokens = 256
batch_size = 16
number_of_epochs = 3
learning_rate = 0.00002
weight_decay = 0.01
frozen_transformer_layers = 0
maximum_tfidf_features = 20_000
# We try each of these values of C for logistic regression. A smaller C means stronger regularization.
tfidf_c_values = [0.1, 1, 10, 100]
decision_threshold = 0.5

# Runtime settings for the one-hour limit
runtime_limit_seconds = 3600
maximum_cpu_threads = 4

In [ ]:
# Create a new output folder for each run.
project_directory = Path.cwd()

run_identifier = datetime.now().strftime("%Y%m%d_%H%M%S")
output_directory = project_directory / "outputs" / run_identifier
output_directory.mkdir(parents=True, exist_ok=False)

# use a GPU if one is available. The NLP VM has only a CPU.
computing_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu_thread_count = min(maximum_cpu_threads, os.cpu_count())
torch.set_num_threads(cpu_thread_count)

# Let's print the main details of this run.
print("PyTorch version:", torch.__version__)
print("Device:", computing_device)
print("CPU threads:", cpu_thread_count)
print("Run outputs:", output_directory)

In [ ]:
def set_random_seeds(random_seed: int):
    """
    This is a helper function to set random seeds for reproducibility across various libraries.
    """
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def save_json(file_path: Path, content: dict):
    """
    This is a helper function to save content as a JSON file.
    """
    file_path.write_text(json.dumps(content, indent=2), encoding="utf-8")

def calculate_metrics(true_labels: np.ndarray, predicted_labels: np.ndarray) -> dict:
    """
    This function claculates various metrics for the text classification task, including
        - precision
        - recall
        - F1 score
        - accuracy
        - confusion matrix counts
    """
    precision, recall, class_f1, support = precision_recall_fscore_support(
        true_labels, predicted_labels, labels=[0, 1], zero_division=0
    )
    confusion_counts = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])
    human_text_count = int(confusion_counts[0].sum())
    return {
        "macro_f1": float(f1_score(true_labels, predicted_labels, labels=[0, 1], average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(true_labels, predicted_labels)),
        "human_precision": float(precision[0]),
        "human_recall": float(recall[0]),
        "human_f1": float(class_f1[0]),
        "ai_generated_precision": float(precision[1]),
        "ai_generated_recall": float(recall[1]),
        "ai_generated_f1": float(class_f1[1]),
        "human_false_positive_rate": float(confusion_counts[0, 1] / human_text_count) if human_text_count else None,
        "human_support": int(support[0]),
        "ai_generated_support": int(support[1]),
    }

# Set random seed
set_random_seeds(SEED)

## 2. Data and model files

We download the HC3 dataset and the DistilBERT model from Hugging Face. The runtime timer starts after the download.

In [ ]:
# Set the variables for the dataset and model download
dataset_repository = "Hello-SimpleAI/HC3"
model_repository = "distilbert/distilbert-base-uncased"

# We set this to True after the first download to run without internet.
local_files_only = False

download_start_time = time.perf_counter()

# Download the dataset from Hugging Face Hub
dataset_path = Path(hf_hub_download(
    repo_id=dataset_repository,
    filename="all.jsonl",
    repo_type="dataset",
    local_files_only=local_files_only,
    token=False,
))

# Download the pretrained DistilBERT model from Hugging Face Hub
pretrained_distilbert_model_directory = snapshot_download(
    repo_id=model_repository,
    allow_patterns=["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json", "vocab.txt"],
    local_files_only=local_files_only,
    token=False,
)

# Load the tokenizer for the pretrained DistilBERT model
tokenizer = AutoTokenizer.from_pretrained(pretrained_distilbert_model_directory, use_fast=True, local_files_only=True)

# Make sure the tokenizer is a fast tokenizer
assert tokenizer.is_fast
# Calculate the time taken for downloading the dataset and model
download_seconds = time.perf_counter() - download_start_time

# We start the runtime timer after the download, so this download timeis not included in the one-hour limit.
run_start_time = time.perf_counter()

# Read the dataset file and parse each line as a list of JSON records
with dataset_path.open(encoding="utf-8") as dataset_file:
    dataset_records = [json.loads(line) for line in dataset_file if line.strip()]

print("Raw question records:", len(dataset_records))

## 3. Prepare paired answers

For each question, we pick one human answer and one ChatGPT answer.
We keep only the first 254 tokens of each answer, so both models see the same text.
DistilBERT adds two special tokens, which gives an input of 256 tokens.
We also remove duplicate questions and answers.

In [ ]:
def normalize_text(text:str) -> str:
    """
    This function normalizes the input text by applying Unicode normalization 
    and replacing repeated whitespace with a single space.
    """
    return " ".join(unicodedata.normalize("NFKC", text).split())

def choose_answer_pairs(records:list, random_seed:int) -> tuple:
    """
    This function processes a list of records containing questions and their corresponding human and AI-generated answers.
    It selects one answer from each class (human and AI-generated) for each unique question.
    """
    answer_generator = random.Random(random_seed)
    seen_questions = set()
    answer_pairs = []
    duplicate_questions = 0
    skipped_records = 0

    for record_index, record in enumerate(records):
        question_text = normalize_text(record["question"]).casefold()
        if not question_text:
            skipped_records += 1
            continue
        if question_text in seen_questions:
            duplicate_questions += 1
            continue
        seen_questions.add(question_text)
        human_answers = [normalize_text(answer) for answer in record["human_answers"] if normalize_text(answer)]
        ai_generated_answers = [normalize_text(answer) for answer in record["chatgpt_answers"] if normalize_text(answer)]
        if not human_answers or not ai_generated_answers:
            skipped_records += 1
            continue
        answer_pairs.append({
            "question_id": record_index,
            "source": record["source"],
            "answers": [answer_generator.choice(human_answers), answer_generator.choice(ai_generated_answers)],
        })
    return answer_pairs, duplicate_questions, skipped_records

def cut_passages_and_remove_duplicates(answer_pairs, text_tokenizer, maximum_input_tokens):
    """
    This function processes a list of answer pairs, tokenizes the answers, and removes duplicates or empty passages.
    """
    # DistilBERT adds two special tokens, so we keep two fewer content tokens than the input limit.
    content_token_limit = maximum_input_tokens - text_tokenizer.num_special_tokens_to_add(pair=False)
    answer_texts = [answer for answer_pair in answer_pairs for answer in answer_pair["answers"]]
    tokenized_answers = text_tokenizer(
        answer_texts, add_special_tokens=False, truncation=True,
        max_length=content_token_limit, return_offsets_mapping=True,
    )
    prepared_rows = []
    seen_answer_texts = set()
    seen_token_sequences = set()
    removed_pairs = 0

    for pair_index, answer_pair in enumerate(answer_pairs):
        pair_rows = []
        pair_answer_texts = []
        pair_token_sequences = []
        for label, answer_text in enumerate(answer_pair["answers"]):
            answer_index = 2 * pair_index + label
            token_ids = tuple(tokenized_answers["input_ids"][answer_index])
            offsets = tokenized_answers["offset_mapping"][answer_index]
            # The end of the last kept token shows where to cut the original text.
            passage_text = answer_text[:offsets[-1][1]] if offsets else ""
            pair_answer_texts.append(answer_text.casefold())
            pair_token_sequences.append(token_ids)
            pair_rows.append({
                "question_id": answer_pair["question_id"],
                "source": answer_pair["source"],
                "label": label,
                "passage_text": passage_text,
                "content_token_count": len(token_ids),
                "original_word_count": len(answer_text.split()),
                "was_truncated": len(passage_text) < len(answer_text),
            })

        # We drop the whole pair if a passage is empty, both answers are the same, or an answer appeared before.
        has_empty_passage = any(not row["passage_text"] for row in pair_rows)
        has_identical_answers = len(set(pair_answer_texts)) < 2 or len(set(pair_token_sequences)) < 2
        has_repeated_answer = (
            any(answer in seen_answer_texts for answer in pair_answer_texts)
            or any(tokens in seen_token_sequences for tokens in pair_token_sequences)
        )
        if has_empty_passage or has_identical_answers or has_repeated_answer:
            removed_pairs += 1
            continue
        prepared_rows.extend(pair_rows)
        seen_answer_texts.update(pair_answer_texts)
        seen_token_sequences.update(pair_token_sequences)

    return pd.DataFrame(prepared_rows), removed_pairs


answer_pairs, duplicate_questions, skipped_records = choose_answer_pairs(dataset_records, SEED)
prepared_answers, removed_pairs = cut_passages_and_remove_duplicates(answer_pairs, tokenizer, maximum_input_tokens)
preparation_counts = {
    "raw_records": len(dataset_records),
    "duplicate_question_records": duplicate_questions,
    "empty_or_incomplete_records": skipped_records,
    "duplicate_or_empty_passage_pairs": removed_pairs,
    "eligible_question_pairs": len(prepared_answers) // 2,
}
save_json(output_directory / "preparation_counts.json", preparation_counts)
display(pd.Series(preparation_counts, name="Count").to_frame())

## 4. Sample and split by question

We sample 2,500 questions and split them into teh following sets:

    - 80% -> training
    - 10% -> validation
    - 10% -> test

We then add 750 more unseen questions to the test set, which gives 1,000 test questions (2,000 answers).

Both answers to a question stay in the same set. We also plot the answer lengths.

In [ ]:
def split_question_pairs(answer_table:pd.DataFrame, number_of_questions:int, extra_test_questions:int, random_seed:int) -> dict:
    """
    This function splits the answer pairs into training, validation, and test sets based on the number of questions specified.
    It then adds extra unseen questions to the test set.
    It ensures that both answers to a question stay in the same set and that no question or passage appears in two different sets.
    """
    
    question_table = answer_table[["question_id", "source"]].drop_duplicates("question_id")
    if number_of_questions + extra_test_questions > len(question_table):
        raise ValueError("The requested sample exceeds the number of eligible question pairs.")
    selected_questions = question_table.sample(n=number_of_questions, random_state=random_seed)
    training_questions, held_out_questions = train_test_split(
        selected_questions["question_id"], test_size=0.2, random_state=random_seed
    )
    validation_questions, test_questions = train_test_split(
        held_out_questions, test_size=0.5, random_state=random_seed
    )
    question_splits = {
        "training": set(training_questions),
        "validation": set(validation_questions),
        "test": set(test_questions),
    }
    # We add more unseen questions to the test set. The training and validation sets do not change.
    if extra_test_questions > 0:
        unused_questions = question_table[~question_table["question_id"].isin(selected_questions["question_id"])]
        extra_questions = unused_questions.sample(n=extra_test_questions, random_state=random_seed)
        question_splits["test"].update(extra_questions["question_id"])
    split_tables = {
        split_name: answer_table[answer_table["question_id"].isin(question_ids)].reset_index(drop=True)
        for split_name, question_ids in question_splits.items()
    }
    return split_tables


def verify_splits(split_tables):
    """
    This function verifies that the splits of the answer pairs meet the following criteria:
        1. Each question has one human answer and one AI-generated answer.
        2. The number of human answers equals the number of AI-generated answers in each split.
        3. No question or passage appears in two different sets.
    """
    for split_name, answer_table in split_tables.items():
        # Each question has one human answer and one AI-generated answer.
        assert (answer_table.groupby("question_id")["label"].nunique() == 2).all(), split_name
        assert (answer_table["label"] == 0).sum() == (answer_table["label"] == 1).sum(), split_name
    # No question or passage appears in two different sets.
    for first_split, second_split in [("training", "validation"), ("training", "test"), ("validation", "test")]:
        first_table = split_tables[first_split]
        second_table = split_tables[second_split]
        assert set(first_table["question_id"]).isdisjoint(second_table["question_id"]), (first_split, second_split)
        assert set(first_table["passage_text"].str.lower()).isdisjoint(second_table["passage_text"].str.lower()), (first_split, second_split)


split_tables = split_question_pairs(prepared_answers, number_of_questions, extra_test_questions, SEED)
verify_splits(split_tables)
training_answers = split_tables["training"]
validation_answers = split_tables["validation"]
test_answers = split_tables["test"]

split_manifest = pd.concat([answer_table.assign(split=split_name) for split_name, answer_table in split_tables.items()], ignore_index=True)
split_manifest.drop(columns="passage_text").to_csv(output_directory / "split_manifest.csv", index=False)
source_counts = split_manifest.groupby(["split", "source", "label"]).size().unstack("label", fill_value=0)
source_counts.columns = class_names
source_counts.to_csv(output_directory / "source_counts.csv")
display(source_counts)
display(training_answers.groupby("label")[["original_word_count", "content_token_count", "was_truncated"]].mean().rename(index=dict(enumerate(class_names))))
print("Overlap checks passed. Answers per split:", {name: len(table) for name, table in split_tables.items()})

# We plot the answer lengths in the training set. Answers over 500 words go into the last bin.
length_figure, length_axis = plt.subplots(figsize=(7, 4))
for label, class_name in enumerate(class_names):
    word_counts = training_answers.loc[training_answers["label"] == label, "original_word_count"].clip(upper=500)
    length_axis.hist(word_counts, bins=25, range=(0, 500), histtype="step", linewidth=2,
                     color=class_colors[label], label=class_name)
length_axis.set_xlabel("Original answer length in words (last bin: 500 or more)")
length_axis.set_ylabel("Training answers")
length_axis.set_title("Answer length by class in the training set")
length_axis.legend(frameon=False)
length_figure.tight_layout()
length_figure.savefig(output_directory / "answer_lengths.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(length_figure)

## 5. Majority and TF-IDF baselines

The majority baseline always predicts the most common training label.
For TF-IDF, we keep punctuation marks and one-letter words as tokens, because they can carry style information. The default tokenizer of scikit-learn removes them.
We try unigrams and unigrams plus bigrams, each with four values of C, and we keep the setting with the best validation macro-F1.
If two settings have the same score, we keep the first one.

In [ ]:
baseline_start_time = time.perf_counter()
# The majority baseline always predicts the most frequent training label.
training_label_counts = np.bincount(training_answers["label"], minlength=2)
majority_label = int(np.argmax(training_label_counts))
majority_validation_predictions = np.full(len(validation_answers), majority_label)
validation_result_rows = [{
    "model": "Majority baseline",
    **calculate_metrics(validation_answers["label"], majority_validation_predictions),
}]

# This pattern keeps words of any length, and it keeps each punctuation mark as a separate token.
tfidf_token_pattern = r"(?u)\b\w+\b|[^\w\s]"

def create_tfidf_classifier(ngram_range, c_value):
    """
    This function creates a TF-IDF vectorizer and a logistic regression classifier pipeline.
    """
    # We put the TF-IDF features and the logistic regression classifier in one pipeline.
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=ngram_range, min_df=2,
            max_features=maximum_tfidf_features,
            lowercase=True, sublinear_tf=True,
            token_pattern=tfidf_token_pattern,
        )),
        ("classifier", LogisticRegression(C=c_value, solver="liblinear", max_iter=1000, random_state=SEED)),
    ])

logistic_regression_candidates = {}
for ngram_name, ngram_range in [("unigrams", (1, 1)), ("unigrams and bigrams", (1, 2))]:
    for c_value in tfidf_c_values:
        candidate_name = f"TF-IDF {ngram_name}, C={c_value}"
        classifier_pipeline = create_tfidf_classifier(ngram_range, c_value)
        classifier_pipeline.fit(training_answers["passage_text"], training_answers["label"])
        validation_probabilities = classifier_pipeline.predict_proba(validation_answers["passage_text"])[:, 1]
        validation_predictions = (validation_probabilities >= decision_threshold).astype(int)
        validation_result_rows.append({"model": candidate_name, **calculate_metrics(validation_answers["label"], validation_predictions)})
        logistic_regression_candidates[candidate_name] = classifier_pipeline

validation_results = pd.DataFrame(validation_result_rows)
# We select the TF-IDF setting with the best validation macro-F1. The first setting wins if scores are equal.
tfidf_results = validation_results[validation_results["model"] != "Majority baseline"]
selected_baseline_name = tfidf_results.loc[tfidf_results["macro_f1"].idxmax(), "model"]
selected_baseline = logistic_regression_candidates[selected_baseline_name]
joblib.dump(selected_baseline, output_directory / "tfidf_logistic_regression.joblib")
baseline_seconds = time.perf_counter() - baseline_start_time
display(validation_results.round(4))
print("Selected TF-IDF setting:", selected_baseline_name)

## 6. DistilBERT data and training functions

We add a classification layer to DistilBERT and write a simple training loop in PyTorch.

In [ ]:
class AnswerDataset(Dataset):
    def __init__(self, answer_table, text_tokenizer, maximum_input_tokens):
        self.encoded_answers = text_tokenizer(
            answer_table["passage_text"].tolist(), truncation=True,
            padding="max_length", max_length=maximum_input_tokens, return_tensors="pt",
        )
        self.labels = torch.tensor(answer_table["label"].to_numpy(), dtype=torch.long)
        actual_lengths = self.encoded_answers["attention_mask"].sum(dim=1).numpy()
        expected_lengths = answer_table["content_token_count"].to_numpy() + text_tokenizer.num_special_tokens_to_add(pair=False)
        assert np.array_equal(actual_lengths, expected_lengths), "Passage tokenization changed after truncation."

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, answer_index):
        encoded_answer = {name: values[answer_index] for name, values in self.encoded_answers.items()}
        encoded_answer["labels"] = self.labels[answer_index]
        return encoded_answer


def make_data_loader(answer_table, shuffle_answers=False):
    """
    This function creates a DataLoader for the given answer table. It uses the AnswerDataset class to encode the answers and labels.
    """
    dataset = AnswerDataset(answer_table, tokenizer, maximum_input_tokens)
    # A fixed generator makes the shuffle order repeatable.
    shuffle_generator = torch.Generator().manual_seed(SEED)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle_answers, generator=shuffle_generator)


def create_distilbert_classifier():
    """
    This function creates a DistilBERT classifier.
    """
    set_random_seeds(SEED)
    classifier = AutoModelForSequenceClassification.from_pretrained(
        pretrained_distilbert_model_directory, num_labels=2, local_files_only=True,
        id2label={0: "human-written", 1: "AI-generated"},
        label2id={"human-written": 0, "AI-generated": 1},
        use_safetensors=True,
    )
    # We can freeze the embeddings and lower layers to reduce training time.
    if frozen_transformer_layers > 0:
        for parameter in classifier.distilbert.embeddings.parameters():
            parameter.requires_grad = False
        for transformer_layer in classifier.distilbert.transformer.layer[:frozen_transformer_layers]:
            for parameter in transformer_layer.parameters():
                parameter.requires_grad = False
    return classifier.to(computing_device)

def create_optimizer(classifier):
    """
    This function creates an AdamW optimizer for the given classifier with specified learning rate and weight decay
    """
    return torch.optim.AdamW(classifier.parameters(), lr=learning_rate, weight_decay=weight_decay)

def train_one_epoch(classifier, data_loader, optimizer):
    """
    This function trains the DistilBERT classifier for one epoch using the provided data loader and optimizer.
    """
    classifier.train()
    total_loss = 0.0
    total_answers = 0
    for batch_index, answer_batch in enumerate(data_loader):
        device_batch = {name: values.to(computing_device) for name, values in answer_batch.items()}
        optimizer.zero_grad()
        model_output = classifier(**device_batch)
        model_output.loss.backward()
        # We clip large gradients to keep training stable.
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
        optimizer.step()
        batch_answer_count = len(device_batch["labels"])
        total_loss += model_output.loss.item() * batch_answer_count
        total_answers += batch_answer_count
        # We show progress during CPU training.
        if (batch_index + 1) % 25 == 0:
            print("Training batches completed:", batch_index + 1, "of", len(data_loader), flush=True)
    return total_loss / total_answers


def predict_ai_generated_probabilities(classifier, data_loader):
    """
    This is the inference function for the DistilBERT classifier. It returns the predicted probabilities of the AI-generated class for each answer in the data loader.
    """
    classifier.eval()
    probability_batches = []
    with torch.no_grad():
        for answer_batch in data_loader:
            model_inputs = {name: values.to(computing_device) for name, values in answer_batch.items() if name != "labels"}
            logits = classifier(**model_inputs).logits
            # We keep the probability of class 1, AI-generated.
            probability_batches.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.concatenate(probability_batches)


training_loader = make_data_loader(training_answers, shuffle_answers=True)
validation_loader = make_data_loader(validation_answers)

## 7. Fine-tune and select the checkpoint

We train DistilBERT for up to three epochs and keep the epoch with the best validation macro-F1.
Then we plot the training curve.

In [ ]:
distilbert_start_time = time.perf_counter()
distilbert_classifier = create_distilbert_classifier()
optimizer = create_optimizer(distilbert_classifier)
best_validation_f1 = -1.0
best_epoch_number = None
training_history = []

for epoch_number in range(1, number_of_epochs + 1):
    epoch_start_time = time.perf_counter()
    training_loss = train_one_epoch(distilbert_classifier, training_loader, optimizer)
    validation_probabilities = predict_ai_generated_probabilities(distilbert_classifier, validation_loader)
    validation_predictions = (validation_probabilities >= decision_threshold).astype(int)
    validation_loss = log_loss(validation_answers["label"], validation_probabilities, labels=[0, 1])
    epoch_metrics = calculate_metrics(validation_answers["label"], validation_predictions)
    if epoch_metrics["macro_f1"] > best_validation_f1:
        best_validation_f1 = epoch_metrics["macro_f1"]
        best_epoch_number = epoch_number
        # We keep a copy of the weights of the best epoch so far.
        best_model_weights = copy.deepcopy(distilbert_classifier.state_dict())
        best_validation_metrics = epoch_metrics
    training_history.append({
        "epoch": epoch_number, "training_loss": training_loss, "validation_loss": validation_loss,
        "validation_macro_f1": epoch_metrics["macro_f1"],
        "epoch_seconds": time.perf_counter() - epoch_start_time,
    })
    # We print the elapsed time so we can check the one-hour limit during the run.
    elapsed_minutes = (time.perf_counter() - run_start_time) / 60
    print("Completed epoch", epoch_number, "| validation macro-F1:", round(epoch_metrics["macro_f1"], 4),
          "| minutes since start:", round(elapsed_minutes, 1))

# We load the best epoch before the test evaluation.
distilbert_classifier.load_state_dict(best_model_weights)
pd.DataFrame(training_history).to_csv(output_directory / "training_history.csv", index=False)
validation_results = pd.concat([
    validation_results,
    pd.DataFrame([{"model": "DistilBERT", **best_validation_metrics}]),
], ignore_index=True)
validation_results.to_csv(output_directory / "validation_metrics.csv", index=False)
save_json(output_directory / "model_selection.json", {
    "selected_tfidf_setting": selected_baseline_name,
    "selected_distilbert_epoch": best_epoch_number,
    "decision_threshold": decision_threshold,
})
distilbert_training_seconds = time.perf_counter() - distilbert_start_time
display(pd.DataFrame(training_history))
display(validation_results.round(4))

In [ ]:
# plot the training curve. The circled point is the epoch we selected.
history_table = pd.DataFrame(training_history)
curve_figure, (loss_axis, f1_axis) = plt.subplots(1, 2, figsize=(11, 4))
loss_axis.plot(history_table["epoch"], history_table["training_loss"], marker="o", markersize=7,
               markeredgecolor="white", linewidth=2, color=training_color, label="Training loss")
loss_axis.plot(history_table["epoch"], history_table["validation_loss"], marker="o", markersize=7,
               markeredgecolor="white", linewidth=2, color=validation_color, label="Validation loss")
loss_axis.set_title("Training and validation loss")
loss_axis.set_ylabel("Cross-entropy loss")
loss_axis.legend(frameon=False)

best_f1 = history_table.loc[history_table["epoch"] == best_epoch_number, "validation_macro_f1"].iloc[0]
f1_axis.plot(history_table["epoch"], history_table["validation_macro_f1"], marker="o", markersize=7,
             markeredgecolor="white", linewidth=2, color=validation_color)
f1_axis.plot([best_epoch_number], [best_f1], marker="o", markersize=15, markerfacecolor="none",
             markeredgecolor="black", markeredgewidth=1.5, linestyle="none")
f1_axis.set_title(f"Validation macro-F1 (selected epoch: {best_epoch_number})")
f1_axis.set_ylabel("Macro-F1")

for plot_axis in [loss_axis, f1_axis]:
    plot_axis.set_xlabel("Epoch")
    plot_axis.set_xticks(history_table["epoch"])
curve_figure.tight_layout()
curve_figure.savefig(output_directory / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(curve_figure)

## 8. Final test evaluation

We test the three models on the same test set.
We report macro-F1, accuracy, the share of human texts labelled as AI, and ROC-AUC.
We also compute a 95% confidence interval for the difference between DistilBERT and TF-IDF.

In [ ]:
evaluation_start_time = time.perf_counter()
test_loader = make_data_loader(test_answers)
test_predictions = {
    "Majority baseline": np.full(len(test_answers), majority_label),
}
test_ai_generated_probabilities = {
    "TF-IDF logistic regression": selected_baseline.predict_proba(test_answers["passage_text"])[:, 1],
    "DistilBERT": predict_ai_generated_probabilities(distilbert_classifier, test_loader),
}
for model_name, ai_generated_probabilities in test_ai_generated_probabilities.items():
    test_predictions[model_name] = (ai_generated_probabilities >= decision_threshold).astype(int)

test_results = pd.DataFrame([
    {"model": model_name, **calculate_metrics(test_answers["label"], predictions)}
    for model_name, predictions in test_predictions.items()
])
test_results.to_csv(output_directory / "test_metrics.csv", index=False)
display(test_results.round(4))

# plot one confusion matrix for each model.
confusion_figure, confusion_axes = plt.subplots(1, 3, figsize=(14, 4))
for plot_axis, (model_name, predictions) in zip(confusion_axes, test_predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        test_answers["label"], predictions, labels=[0, 1], display_labels=class_names,
        ax=plot_axis, colorbar=False, cmap="Blues",
    )
    plot_axis.set_title(model_name)
    plot_axis.grid(False)
confusion_figure.tight_layout()
confusion_figure.savefig(output_directory / "confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(confusion_figure)

baseline_test_f1 = test_results.set_index("model").loc["TF-IDF logistic regression", "macro_f1"]
distilbert_test_f1 = test_results.set_index("model").loc["DistilBERT", "macro_f1"]
# ROC-AUC does not depend on the 0.5 threshold, so we report it as an extra metric.
for model_name, ai_generated_probabilities in test_ai_generated_probabilities.items():
    print(model_name, "test ROC-AUC:", round(roc_auc_score(test_answers["label"], ai_generated_probabilities), 4))

# Etsimate a 95% confidence interval for the difference by resampling the test answers 2,000 times.
test_labels = test_answers["label"].to_numpy()
bootstrap_generator = np.random.default_rng(SEED)
bootstrap_differences = []
for _ in range(2000):
    sample_indices = bootstrap_generator.integers(0, len(test_labels), len(test_labels))
    distilbert_sample_f1 = f1_score(test_labels[sample_indices], test_predictions["DistilBERT"][sample_indices],
                                    labels=[0, 1], average="macro", zero_division=0)
    baseline_sample_f1 = f1_score(test_labels[sample_indices], test_predictions["TF-IDF logistic regression"][sample_indices],
                                  labels=[0, 1], average="macro", zero_division=0)
    bootstrap_differences.append(distilbert_sample_f1 - baseline_sample_f1)
lower_bound, upper_bound = np.percentile(bootstrap_differences, [2.5, 97.5])
save_json(output_directory / "test_difference.json", {
    "distilbert_minus_tfidf_macro_f1": float(distilbert_test_f1 - baseline_test_f1),
    "bootstrap_95_percent_interval": [float(lower_bound), float(upper_bound)],
})
print(f"DistilBERT minus TF-IDF macro-F1 on the test set: {distilbert_test_f1 - baseline_test_f1:.4f}")
print(f"95% bootstrap confidence interval: [{lower_bound:.4f}, {upper_bound:.4f}]")

# Compare the three models on the two main test metrics.
comparison_figure, comparison_axes = plt.subplots(1, 2, figsize=(11, 4))
model_tick_labels = [model_name.replace(" ", "\n", 1) for model_name in test_results["model"]]
bar_colors = [model_colors[model_name] for model_name in test_results["model"]]
for plot_axis, metric_name, axis_title in zip(
    comparison_axes,
    ["macro_f1", "human_false_positive_rate"],
    ["Test macro-F1 (higher is better)", "Human texts labelled AI (lower is better)"],
):
    bars = plot_axis.bar(model_tick_labels, test_results[metric_name], width=0.5, color=bar_colors)
    plot_axis.bar_label(bars, fmt="%.3f", padding=3)
    plot_axis.set_ylim(0, test_results[metric_name].max() * 1.2)
    plot_axis.set_title(axis_title)
comparison_figure.tight_layout()
comparison_figure.savefig(output_directory / "test_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(comparison_figure)

## 9. Error analysis

We look at the results by source and by answer length, check some wrong predictions, and list the most important TF-IDF words.
We also check how well answer length alone can predict the label.

In [ ]:
prediction_rows = []
for model_name, predictions in test_predictions.items():
    model_predictions = test_answers.copy()
    model_predictions["model"] = model_name
    model_predictions["predicted_label"] = predictions
    model_predictions["ai_generated_probability"] = test_ai_generated_probabilities.get(model_name, np.full(len(test_answers), np.nan))
    prediction_rows.append(model_predictions)
prediction_table = pd.concat(prediction_rows, ignore_index=True)
prediction_table.drop(columns="passage_text").to_csv(output_directory / "test_predictions.csv", index=False)
# We compare full answers with answers cut at the token limit, because most AI answers are cut.
prediction_table["length_group"] = np.where(prediction_table["was_truncated"], "cut at token limit", "full answer")

slice_result_rows = []
for grouping_column in ["source", "length_group"]:
    for (model_name, group_name), group_answers in prediction_table.groupby(["model", grouping_column], observed=True):
        slice_result_rows.append({
            "model": model_name, "grouping": grouping_column, "group": str(group_name),
            "answer_count": len(group_answers),
            **calculate_metrics(group_answers["label"], group_answers["predicted_label"]),
        })
slice_results = pd.DataFrame(slice_result_rows)
slice_results.to_csv(output_directory / "test_slice_metrics.csv", index=False)
display(slice_results[["model", "grouping", "group", "answer_count", "macro_f1", "human_false_positive_rate"]].round(4))

def plot_model_bars_by_group(plot_axis, grouping_column, metric_name, count_column, count_word):
    """
    This function plots bar charts for each learned model in each group, showing the specified metric and group size.
    """
    
    grouping_rows = slice_results[slice_results["grouping"] == grouping_column]
    group_names = list(grouping_rows["group"].unique())
    bar_width = 0.38
    model_bars = []
    for model_index, model_name in enumerate(["TF-IDF logistic regression", "DistilBERT"]):
        model_rows = grouping_rows[grouping_rows["model"] == model_name].set_index("group").loc[group_names]
        bar_positions = np.arange(len(group_names)) + (model_index - 0.5) * bar_width
        model_bars.append(plot_axis.bar(bar_positions, model_rows[metric_name], width=bar_width * 0.95,
                                        color=model_colors[model_name], label=model_name))
    group_counts = grouping_rows.drop_duplicates("group").set_index("group").loc[group_names, count_column]
    tick_labels = [f"{group_name}\n({group_count} {count_word})" for group_name, group_count in zip(group_names, group_counts)]
    plot_axis.set_xticks(np.arange(len(group_names)), tick_labels)
    return model_bars


# plot macro-F1 by source, and the human-text false-positive rate for full and cut answers.
slice_figure, (source_axis, truncation_axis) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [5, 3]})
plot_model_bars_by_group(source_axis, "source", "macro_f1", "answer_count", "answers")
source_axis.set_ylim(0, 1.2)
source_axis.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
source_axis.set_ylabel("Macro-F1")
source_axis.set_title("Test macro-F1 by source")
source_axis.legend(frameon=False, loc="upper left", ncol=2)
truncation_bars = plot_model_bars_by_group(truncation_axis, "length_group", "human_false_positive_rate", "human_support", "human")
# write the values on these bars, because a rate of zero has no visible bar.
for bars in truncation_bars:
    truncation_axis.bar_label(bars, fmt="%.3f", padding=3)
truncation_axis.margins(y=0.15)
truncation_axis.set_ylabel("Share of human texts labelled AI")
truncation_axis.set_title("Human texts labelled AI: full vs cut answers")
slice_figure.tight_layout()
slice_figure.savefig(output_directory / "test_f1_by_group.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(slice_figure)

# check how well answer length alone separates the classes, because length is a possible shortcut.
length_only_classifier = LogisticRegression()
length_only_classifier.fit(training_answers[["content_token_count"]], training_answers["label"])
for split_name, answer_table in [("validation", validation_answers), ("test", test_answers)]:
    length_only_predictions = length_only_classifier.predict(answer_table[["content_token_count"]])
    length_only_f1 = f1_score(answer_table["label"], length_only_predictions, average="macro")
    print(f"Length-only reference, {split_name} macro-F1:", round(length_only_f1, 4))

# Show the five most confident mistakes of each kind for the two learned models.
error_examples = []
for model_name in ["TF-IDF logistic regression", "DistilBERT"]:
    model_rows = prediction_table[prediction_table["model"] == model_name]
    human_errors = model_rows[(model_rows["label"] == 0) & (model_rows["predicted_label"] == 1)]
    human_errors = human_errors.sort_values("ai_generated_probability", ascending=False).head(5)
    error_examples.append(human_errors.assign(error_type="Human text labelled AI"))
    ai_generated_errors = model_rows[(model_rows["label"] == 1) & (model_rows["predicted_label"] == 0)]
    ai_generated_errors = ai_generated_errors.sort_values("ai_generated_probability").head(5)
    error_examples.append(ai_generated_errors.assign(error_type="AI text labelled human"))
error_examples = pd.concat(error_examples, ignore_index=True)
error_examples.to_csv(output_directory / "error_examples.csv", index=False)
display(error_examples[["model", "error_type", "source", "ai_generated_probability", "passage_text"]])

# Negative coefficients point to human-written text and positive coefficients point to AI-generated text.
feature_names = selected_baseline.named_steps["tfidf"].get_feature_names_out()
feature_coefficients = selected_baseline.named_steps["classifier"].coef_[0]
ordered_feature_indices = np.argsort(feature_coefficients)
important_features = pd.DataFrame({
    "human_associated_feature": feature_names[ordered_feature_indices[:10]],
    "human_coefficient": feature_coefficients[ordered_feature_indices[:10]],
    "ai_generated_associated_feature": feature_names[ordered_feature_indices[-10:][::-1]],
    "ai_generated_coefficient": feature_coefficients[ordered_feature_indices[-10:][::-1]],
})
important_features.to_csv(output_directory / "logistic_regression_features.csv", index=False)
display(important_features)

## 10. Save the results

Save the runtime of each part of the run.

In [ ]:
evaluation_and_analysis_seconds = time.perf_counter() - evaluation_start_time
total_runtime_seconds = time.perf_counter() - run_start_time
runtime_summary = {
    "download_or_cache_lookup_seconds": download_seconds,
    "baseline_seconds": baseline_seconds,
    "distilbert_training_and_selection_seconds": distilbert_training_seconds,
    "test_evaluation_and_analysis_seconds": evaluation_and_analysis_seconds,
    "total_runtime_seconds": total_runtime_seconds,
    "total_runtime_with_download_seconds": total_runtime_seconds + download_seconds,
    "runtime_limit_seconds": runtime_limit_seconds,
    "within_runtime_limit": total_runtime_seconds <= runtime_limit_seconds,
    "question_count": number_of_questions,
    "extra_test_questions": extra_test_questions,
    "test_answer_count": len(test_answers),
    "selected_distilbert_epoch": best_epoch_number,
    "device": str(computing_device),
}
save_json(output_directory / "runtime_summary.json", runtime_summary)
display(pd.Series(runtime_summary, name="Run summary").to_frame())
print("We saved this run in:", output_directory)
print("Total runtime in minutes:", round(total_runtime_seconds / 60, 1))
